In [1]:
import IPython
display(IPython.display.Javascript('''
(function(){
  if (window.__keepAliveTimer) {
    console.log("[KeepAlive] already running");
    return;
  }
  function SimulateActivity() {
    const timestamp = new Date().toLocaleTimeString();
    document.dispatchEvent(new MouseEvent('mousemove', { bubbles: true }));
    document.dispatchEvent(new KeyboardEvent('keydown', { bubbles: true, key: 'Shift' }));
    console.log("[KeepAlive] activity ping @ " + timestamp);
  }
  window.__keepAliveTimer = setInterval(SimulateActivity, 60000);
  console.log("[KeepAlive] started");
})();
'''))

<IPython.core.display.Javascript object>

In [ ]:
import IPython
display(IPython.display.Javascript('''
if (window.__keepAliveTimer) {
  clearInterval(window.__keepAliveTimer);
  window.__keepAliveTimer = null;
  console.log("[KeepAlive] stopped");
} else {
  console.log("[KeepAlive] was not running");
}
'''))

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Persiapan Library

In [3]:
!pip install ultralytics roboflow gradio

import os
import matplotlib.pyplot as plt
import pandas as pd
from roboflow import Roboflow
from ultralytics import YOLO
from IPython.display import Image, display
import gradio as gr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-

# Download Dataset dari Roboflow

In [4]:
print("Mengunduh dataset dari Roboflow...")

from google.colab import userdata
from roboflow import Roboflow

api_key_roboflow = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key_roboflow)

project = rf.workspace("caesar-ylvsf").project("hair-2pjwo-ng8p4")
version = project.version(1)
dataset = version.download("yolov8")

dataset_path = dataset.location
path_ke_yaml = os.path.join(dataset_path, "data.yaml")

Mengunduh dataset dari Roboflow...
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to hair-1 in yolov8:: 100%|██████████| 6601/6601 [00:01<00:00, 5772.61it/s]


# Mulai Train Model

In [5]:
print("\nMemulai training model segmentasi...")

model = YOLO("yolo26n-seg.pt")

results = model.train(
    data=path_ke_yaml,
    imgsz=640,
    epochs=50,
    save_period=5,        # simpan checkpoint tiap x epoch
    project="/content/drive/MyDrive/ITC/seg-hair/Baseline",  # tersimpan meski VM reset
    patience=8,
    batch=32,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.001,
    cos_lr=True,
    weight_decay=0.0006,
    overlap_mask=True,
    mask_ratio=2,         # naikkan presisi GT mask dari default 4, cocok untuk objek berdetail seperti rambut
    retina_masks=True,    # mask output lebih halus saat val/predict/demo
    name="Hair-Segment-Base",
    device=0
)

folder_hasil = "/content/drive/MyDrive/ITC/seg-hair/Baseline/Hair-Segment-base"


Memulai training model segmentasi...
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/hair-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.001, mask_ratio=2, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0

# Menampilkan Grafik Hasil Training

In [7]:
print("\n--- GRAFIK PERFORMA MODEL ---")
path_grafik = os.path.join(folder_hasil, "results.png")

if os.path.exists(path_grafik):
    display(Image(filename=path_grafik, width=800))
else:
    print("Grafik performa tidak ditemukan.")


--- GRAFIK PERFORMA MODEL ---
Grafik performa tidak ditemukan.


# Custom Graph

In [8]:
results_dir = folder_hasil
df = pd.read_csv(f"{results_dir}/results.csv")
df.columns = df.columns.str.strip()  # header CSV Ultralytics sering ada spasi di depan

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Loss (box + seg + cls + dfl) -- 'seg_loss' adalah komponen khusus segmentasi (mask loss)
train_loss = df['train/box_loss'] + df['train/seg_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
axes[0].plot(df['epoch'], train_loss, label='Train Loss', linewidth=2.5, color='#e74c3c')
if all(c in df.columns for c in ['val/box_loss', 'val/seg_loss', 'val/cls_loss', 'val/dfl_loss']):
    val_loss = df['val/box_loss'] + df['val/seg_loss'] + df['val/cls_loss'] + df['val/dfl_loss']
    axes[0].plot(df['epoch'], val_loss, label='Val Loss', linewidth=2.5, color='#3498db')
axes[0].set_xlabel('Epoch', fontsize=13)
axes[0].set_ylabel('Loss', fontsize=13)
axes[0].set_title('Loss (box + seg + cls + dfl)', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# mAP50 -- (B) untuk bounding box, (M) untuk mask segmentasi
if 'metrics/mAP50(B)' in df.columns:
    axes[1].plot(df['epoch'], df['metrics/mAP50(B)'] * 100, label='mAP50 (Box)', linewidth=2.5, color='#f39c12')
if 'metrics/mAP50(M)' in df.columns:
    axes[1].plot(df['epoch'], df['metrics/mAP50(M)'] * 100, label='mAP50 (Mask)', linewidth=2.5, color='#2ecc71')
axes[1].set_xlabel('Epoch', fontsize=13)
axes[1].set_ylabel('mAP50 (%)', fontsize=13)
axes[1].set_title('Validation mAP50', fontsize=15, fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Hair-Segment-Base Training Results', fontsize=17, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f"{results_dir}/results_readable.png", dpi=150, bbox_inches='tight')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ITC/seg-hair/Baseline/Hair-Segment-base/results.csv'

# Confusion Matrix & Kurva Performa Tambahan

In [9]:
kurva = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxPR_curve.png",
    "MaskF1_curve.png",
    "MaskP_curve.png",
    "MaskR_curve.png",
    "MaskPR_curve.png",
]

print("\nHASIL VALIDASI: CONFUSION MATRIX & KURVA PERFORMA\n")

displayed = set()
for img in kurva:
    path = os.path.join(folder_hasil, img)
    if os.path.exists(path) and img not in displayed:
        print(f"Displaying {img}")
        display(Image(path))
        displayed.add(img)
    else:
        print(f"Warning: {img} tidak ditemukan di {folder_hasil}.")


HASIL VALIDASI: CONFUSION MATRIX & KURVA PERFORMA



# Visualisasi Dengan Data Test 1-by-1

In [ ]:
model = YOLO(f"{folder_hasil}/weights/best.pt")

test_folder = f"{dataset_path}/test/images"

for file in os.listdir(test_folder):
    if file.endswith((".jpg", ".png", ".jpeg")):
        image_path = os.path.join(test_folder, file)
        hasil = model(image_path, verbose=False)

        r = hasil[0]
        if r.masks is not None:
            annotated = r.plot(boxes=False, labels=True)
            title_info = f"{len(r.masks)} objek tersegmentasi"
        else:
            annotated = r.plot()
            title_info = "tidak ada objek terdeteksi"

        plt.figure(figsize=(6, 6))
        plt.imshow(annotated[..., ::-1])
        plt.axis("off")
        plt.title(f"{file} | {title_info}")
        plt.show()

# Visualisasi Dengan Data Test Batch

In [ ]:
model = YOLO(f"{folder_hasil}/weights/best.pt")

metrics = model.val(
    data=path_ke_yaml,
    split='test',
    project="/content/drive/MyDrive/ITC/seg-hair/Baseline",
    name="Hair-Segment-Base-test"
)

print(f"Test Box    mAP50    : {metrics.box.map50:.2%}")
print(f"Test Box    mAP50-95 : {metrics.box.map:.2%}")
print(f"Test Box    Precision: {metrics.box.mp:.2%}")
print(f"Test Box    Recall   : {metrics.box.mr:.2%}")
print()
print(f"Test Mask   mAP50    : {metrics.seg.map50:.2%}")
print(f"Test Mask   mAP50-95 : {metrics.seg.map:.2%}")
print(f"Test Mask   Precision: {metrics.seg.mp:.2%}")
print(f"Test Mask   Recall   : {metrics.seg.mr:.2%}")

# Coba dengan Gradio

In [10]:
model = YOLO(f"{folder_hasil}/weights/best.pt")

def predict(image):
    results = model(image, verbose=False)
    r = results[0]

    if r.masks is not None:
        annotated = r.plot(boxes=False, labels=True)
    else:
        annotated = r.plot()

    return annotated[..., ::-1]  # BGR -> RGB untuk tampilan Gradio

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Image(type="numpy"),
    title="Hair Segmentation Detector",
    description="Unggah gambar rambut untuk melihat hasil segmentasi (mask) objek."
)

demo.launch()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ITC/seg-hair/Baseline/Hair-Segment-base/weights/best.pt'

# Export Model untuk Deployment

In [ ]:
model = YOLO(f"{folder_hasil}/weights/best.pt")

# Desktop, Server, Windows, Linux, Web, C++
model.export(format="onnx")

# Android, Raspberry Pi, Embedded, Device, IoT
model.export(format="tflite")

# Android
model.export(format="ncnn")

# iPhone, iPad, MacBook, Vision Pro, Apple Watch
model.export(format="coreml")